### Historical Prediction Table

race_id, driver, predicted_points, actual_points, residual

#### Create Bucket feature
Instead of grouping by driver, group by the size of the prediction.

Example buckets
0-5: low
5-10: cheap/low-mid
10-15: mid
15-20: strong
20+ elite

In [29]:
import duckdb
from pathlib import Path

In [30]:
working_directory = Path.cwd().parent.parent
DATABASE_PATH = working_directory / "data" / "database" / "f1_fantasy.duckdb"

In [31]:
def read_dpr():
    with duckdb.connect(DATABASE_PATH, read_only=True) as con:
        result = con.execute("""Select * FROM driver_prediction_residuals""").df()
        return result

In [32]:
dpr = read_dpr()
dpr.tail(20)

,prediction_run_id,model_name,model_version,feature_set_version,target,year,race_id,driver_id,constructor_id,price,is_sprint_weekend,actual_points,predicted_points,residual,abs_residual,prediction_bucket
44,3,MVP,2,2,points,2026,73,38,16,22.9,False,19.0,22.772778,-3.772778,3.772778,20_plus
45,3,MVP,2,2,points,2026,73,18,6,7.0,False,4.0,3.942200,0.057800,0.057800,00_20
46,3,MVP,2,2,points,2026,73,15,20,8.5,False,9.0,6.525777,2.474223,2.474223,00_20
47,3,MVP,2,2,points,2026,73,9,41,12.2,False,4.0,7.763332,-3.763332,3.763332,00_20
48,4,MVP,3,2,points,2026,76,9,41,12.4,True,19.0,8.309003,10.690997,10.690997,00_20
49,4,MVP,3,2,points,2026,76,39,33,7.5,True,-12.0,5.180960,-17.180960,17.180960,00_20
50,4,MVP,3,2,points,2026,76,36,26,26.5,True,54.0,21.830826,32.169174,32.169174,20_plus
51,4,MVP,3,2,points,2026,76,2,41,10.2,True,20.0,4.328561,15.671439,15.671439,00_20
52,4,MVP,3,2,points,2026,76,34,30,24.1,True,42.0,28.103493,13.896507,13.896507,20_plus
53,4,MVP,3,2,points,2026,76,38,16,23.2,True,20.0,24.644225,-4.644225,4.644225,20_plus


In [33]:

def read_driver_residual_bucket_stats():
    with duckdb.connect(DATABASE_PATH, read_only=True) as con:
        result = con.execute("""


            SELECT
                prediction_bucket,

                COUNT(*) AS sample_size,

                AVG(residual) AS mean_residual,
                STDDEV(residual) AS residual_std,

                QUANTILE_CONT(residual, 0.05) AS p05,
                QUANTILE_CONT(residual, 0.10) AS p10,
                QUANTILE_CONT(residual, 0.25) AS p25,
                QUANTILE_CONT(residual, 0.50) AS median,
                QUANTILE_CONT(residual, 0.75) AS p75,
                QUANTILE_CONT(residual, 0.90) AS p90,
                QUANTILE_CONT(residual, 0.95) AS p95

            FROM driver_prediction_residuals

            GROUP BY prediction_bucket

            ORDER BY prediction_bucket;
                             

                             """).df()
        return result

In [34]:
bucket_stats = read_driver_residual_bucket_stats()
bucket_stats

,prediction_bucket,sample_size,mean_residual,residual_std,p05,p10,p25,median,p75,p90,p95
0,00_20,43,1.404136,16.131502,-23.661863,-20.491846,-7.205595,2.474223,9.652507,15.641849,24.795831
1,20_plus,21,6.037656,17.660773,-27.015713,-21.295369,-3.772778,10.105486,14.491791,26.658292,28.928456


### reading the risk summary

In [41]:
import pandas as pd
PATH = working_directory / "driver_risk_summary_5_15.csv"
driver_summary = pd.read_csv(PATH)

In [ ]:
driver_summary.sort_values("mean_sim_points", ascending=False)

,Unnamed: 0,driver_id,driver_name,constructor_id,constructor_name,price,predicted_points,mean_sim_points,std_sim_points,p05,...,p25,median,p75,p90,p95,prediction_bucket,p90_per_price,mean_per_price,risk_range,downside_gap
21,20,60,sergio perez,12,Cadillac,7.6,2.563696,3.798358,15.865939,-21.355876,...,-6.321717,5.037919,11.177713,18.235135,28.186353,00_20,2.399360,0.499784,36.682557,22.245780
20,21,65,valtteri bottas,12,Cadillac,3.9,3.188463,4.663307,15.953585,-20.731109,...,-5.696951,5.662685,13.879460,18.859902,28.811120,00_20,4.835872,1.195720,36.682557,22.485962
19,9,26,isack hadjar,37,Red Bull Racing,12.7,4.042558,5.474371,16.161947,-21.395808,...,-4.842855,6.516781,14.733555,19.713997,29.665215,00_20,1.552283,0.431053,36.682557,22.442931
16,16,44,nico hulkenberg,11,Audi,4.4,4.936139,6.180017,15.931940,-18.983432,...,-3.949274,7.410362,15.627136,20.607578,30.558796,00_20,4.683540,1.404549,36.682557,22.254996
17,14,39,liam lawson,33,Racing Bulls,8.1,4.849901,6.237696,16.268237,-19.069670,...,-4.035512,7.324124,15.540898,20.521340,30.472558,00_20,2.533499,0.770086,36.682557,22.398913
18,6,18,franco colapinto,6,Alpine,8.2,4.786876,6.368598,15.929262,-19.132695,...,-0.738901,7.261099,15.477873,20.458315,30.409533,00_20,2.494916,0.776658,36.682557,22.592839
14,1,6,arvid lindblad,33,Racing Bulls,7.0,5.513928,6.715795,15.954880,-18.405643,...,-3.371485,7.988151,14.127945,21.185367,31.136585,00_20,3.026481,0.959399,36.682557,22.212985
15,4,15,esteban ocon,20,Haas F1 Team,9.7,5.268395,6.750609,15.880823,-18.651176,...,-3.617018,7.742618,15.959392,20.939834,30.891052,00_20,2.158746,0.695939,36.682557,22.493331
13,5,17,fernando alonso,10,Aston Martin,8.0,6.276917,7.781730,16.066540,-17.642654,...,-2.608496,8.751140,16.967914,21.948356,31.899574,00_20,2.743545,0.972716,36.682557,22.515931
10,11,35,lance stroll,10,Aston Martin,5.6,6.619031,7.813074,15.739245,-17.300540,...,-2.266382,9.093254,15.233048,22.290470,32.241688,00_20,3.980441,1.395192,36.682557,22.205160
